# 🚀 PostgreSQL (pgvector) 기반 공통 RAG 검색 파이프라인 가이드 (심화)

본 노트북은 **BIST Mini-2 프로젝트**에서 사용되는 공통 RAG 임베딩 검색 파이프라인의 **LangChain 및 LangGraph 에이전트 연동용 툴(Tools) 인터페이스**를 학습하기 위한 심화 가이드입니다.

RAG 검색 기능이 에이전트 그래프(Agent Graph) 상에서 독립적인 툴로서 바인딩되는 규격과, LangGraph의 상태(State)를 제어하는 `Command` 데이터 흐름을 깊게 알아봅니다.

---

## 📌 심화 연동 핵심 포인트
1. **LangChain `@tool` 데코레이터** : 일반 Python 비동기 함수를 에이전트가 호출할 수 있는 도구 규격으로 래핑합니다.
2. **`ToolRuntime` 컨텍스트** : 툴이 호출된 실시간 세션 정보(`tool_call_id` 등)를 추적하여 LLM의 요청과 툴 메시지(ToolMessage)를 1:1로 매핑합니다.
3. **`Command(update={...})` 그래프 상태 제어** : 에이전트 노드(Node) 실행 후 대화 이력(`messages`)뿐만 아니라, RAG의 인용 출처 데이터 스토어(`sources`)에 논문 인용 메타데이터를 직접 주입(update)합니다.

## 1단계. 환경 및 백엔드 설정 로드

노트북 실행 위치를 설정하고 백엔드 패키지 경로를 주입합니다.

In [7]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# 현재 notebooks/rag_pipeline/ 폴더 경로 기준 백엔드 디렉토리 탐색
current_dir = Path("").resolve() if '__file__' not in locals() else Path(__file__).parent
backend_dir = (current_dir / "../../backend").resolve()

if str(backend_dir) not in sys.path:
    sys.path.append(str(backend_dir))

# 백엔드 .env 로드
env_path = backend_dir / ".env"
if env_path.exists():
    load_dotenv(dotenv_path=env_path)
    print(f"✅ 백엔드 환경 설정 로드 완료: {env_path}")

openai_key = os.getenv("OPENAI_API_KEY", "")
database_url = os.getenv("DATABASE_URL", "")

✅ 백엔드 환경 설정 로드 완료: /Users/pileuszu/Repos/bist-mini-2/backend/.env


## 2단계. LangGraph 툴 테스트용 가상 런타임 환경 구성 (Mock)

실제 LangGraph 프레임워크 환경이 아니더라도 노트북 상에서 툴 함수를 수동으로 테스트할 수 있도록, `ToolRuntime` 규격을 모사하는 가상의 Mock 객체를 설계합니다.

In [8]:
from langchain.tools import ToolRuntime

mock_runtime = ToolRuntime(
    state=None,
    context=None,
    config={},
    stream_writer=lambda _: None,
    tool_call_id="call_mock_12345",
    store=None
)
print(f"✅ Mock ToolRuntime 준비 완료 (tool_call_id = '{mock_runtime.tool_call_id}')")

✅ Mock ToolRuntime 준비 완료 (tool_call_id = 'call_mock_12345')


## 3단계. LangGraph 전용 RAG 검색 툴 호출 시뮬레이션

실서비스 모듈(`api.common.rag_pipeline`)에 선언되어 있는 `@tool` 기반 컴퓨터 과학(CS) 논문 검색 도구인 `search_cs_papers`를 직접 실행해봅니다.

* 이 툴은 일반 데이터를 문자열로 직접 리턴하는 대신, **LangGraph의 상태를 업데이트**하는 `Command` 객체를 반환합니다.

In [9]:
from langchain_openai import OpenAIEmbeddings
from langchain_postgres import PGVector
from langchain.tools import tool, ToolRuntime
from langchain_core.messages import ToolMessage
from langgraph.types import Command

# text-embedding-3-large 모델 고정 규격 사용
EMBED_MODEL = "text-embedding-3-large"
embeddings_model = OpenAIEmbeddings(
    model=EMBED_MODEL,
    api_key=openai_key
)

# 데이터베이스 연결 정보 설정
pgvector_url = database_url.replace("postgresql+asyncpg://", "postgresql+psycopg_async://")

DOMAIN_COLLECTIONS = {
    "bio": "bio_embeddings",
    "cs": "cs_embeddings",
    "astronomy": "astronomy_embeddings"
}


### 3.1단계. 핵심 유사도 검색 함수 정의 (`similarity_search`)

이제 지정된 도메인의 벡터 스페이스(컬렉션)에 접속하여 사용자의 질의(Query)와 유사한 논문 청크를 검색하는 핵심 도우미 함수 `similarity_search`를 작성합니다.

* **`PGVector`**: LangChain의 pgvector 연동 모듈로, 임베딩 모델과 DB 연결 URL, 대상 테이블(Collection) 이름을 받아 초기화됩니다.
* **`asimilarity_search_with_score`**: 비동기로 코사인 거리에 기반한 유사도 검색을 수행하며, `(Document, distance_score)` 튜플 리스트를 반환합니다.
* **코사인 유사도 변환**: 반환된 `score`는 거리 점수(Distance)이므로, `1.0 - score` 연산을 통해 유사도(Similarity)로 환산하여 최종 반환 딕셔너리에 담습니다.
* **안정적인 타입 변환 (`safe_str`)**: 데이터베이스에서 결측치(`None` 또는 `NaN`)가 포함되어 넘어올 경우 문자열 연산 오류가 발생하는 것을 방지하기 위해 모든 텍스트 요소를 안전하게 문자열로 변환합니다.

In [10]:
import math

def safe_str(val):
    if val is None:
        return ""
    if isinstance(val, float) and math.isnan(val):
        return ""
    return str(val)

async def similarity_search(domain: str, query: str, k: int = 3):
    if domain not in DOMAIN_COLLECTIONS:
        raise ValueError(f"지원하지 않는 도메인입니다: {domain}")
    collection_name = DOMAIN_COLLECTIONS[domain]
    vectorstore = PGVector(
        embeddings=embeddings_model,
        collection_name=collection_name,
        connection=pgvector_url,
        async_mode=True,
    )
    results = await vectorstore.asimilarity_search_with_score(query, k=k)
    formatted_results = []
    for doc, score in results:
        meta = doc.metadata or {}
        arxiv_id = meta.get("arxiv_id") or meta.get("doc_id") or ""
        title = meta.get("title", "")
        similarity = round(1.0 - score, 4)
        formatted_results.append({
            "doc_id": safe_str(arxiv_id),
            "title": safe_str(title),
            "text_chunk": safe_str(doc.page_content),
            "score": similarity
        })
    return formatted_results


### 3.2단계. LangGraph 전용 CS 논문 검색 도구 정의 (`search_cs_papers`)

`similarity_search` 검색 헬퍼를 바탕으로, 실제 LangGraph 에이전트가 호출할 수 있는 컴퓨터 과학(cs.NE) 분야 검색 툴을 설계합니다.

* **`@tool` 데코레이터**: 일반 Python 비동기 함수를 LangChain 규격의 툴로 변환합니다. 함수의 Docstring은 LLM이 이 툴을 언제 사용해야 하는지 판단하는 프롬프트 정보로 활용됩니다.
* **`ToolRuntime` 컨텍스트**: 현재 세션의 메타데이터(예: LLM이 보낸 툴 호출 ID인 `tool_call_id`)를 가져오는 데 필요합니다.
* **`Command` 반환 패턴**: 단순한 결과 문자열을 반환하는 대신, `Command(update={...})` 객체를 반환합니다.
  - **`messages`**: 대화 상태 채널에 `ToolMessage`를 담아 추가합니다. LLM은 이 응답을 읽고 답변을 구성합니다.
  - **`sources`**: 대화 메시지와는 별도로, 화면 UI 위젯이나 참고문헌 뷰어에 표시할 출처 데이터(`arxiv_id`, `title`, `summary`)를 전역 그래프 상태에 저장(update)합니다.

In [11]:
@tool
async def search_cs_papers(
    query: str,
    runtime: ToolRuntime,
    k: int = 3
) -> Command:
    """컴퓨터 과학(cs.NE) 관련 학술 논문 데이터베이스에서 관련 내용을 검색합니다."""
    results = await similarity_search("cs", query, k=k)

    if not results:
        msg = f"cs.NE 논문에서 '{query}'와 관련된 내용을 찾을 수 없습니다."
        return Command(update={
            "messages": [ToolMessage(content=msg, tool_call_id=runtime.tool_call_id)],
        })

    output_lines = [f"검색 결과: '{query}' (cs.NE 논문)\n", "=" * 80]
    sources = []
    for idx, r in enumerate(results, 1):
        arxiv_id = r["doc_id"]
        title = r["title"]
        score = r["score"]
        output_lines.append(f"\n[논문 {idx}] (유사도: {score:.4f})")
        output_lines.append(f"제목: {title}")
        output_lines.append(f"arXiv ID: {arxiv_id}")
        output_lines.append(f"\n내용:\n{r['text_chunk']}\n")
        output_lines.append("-" * 80)
        
        # 본문 텍스트가 비어 있거나 문자열이 아닌 경우를 방비
        text_chunk = r.get("text_chunk")
        if not isinstance(text_chunk, str):
            text_chunk = ""
        snippet = " ".join(text_chunk.split())
        if len(snippet) > 160:
            snippet = snippet[:160].rstrip() + "…"
        sources.append({"arxiv_id": arxiv_id, "title": safe_str(title), "summary": snippet})

    tool_text = "\n".join(output_lines)
    return Command(update={
        "messages": [ToolMessage(content=tool_text, tool_call_id=runtime.tool_call_id)],
        "sources": sources,
    })


### 3.3단계. 툴 비동기 호출 테스트

구성한 `search_cs_papers` 툴을 로컬 테스트 쿼리를 입력해 호출해보고 정상적으로 `Command` 객체가 생성되는지 검증합니다.
이 과정에서는 앞서 준비한 `mock_runtime`을 함께 전달하여 LangGraph 프레임워크 밖에서도 툴 호출 생명주기가 매핑되도록 지원합니다.

In [12]:
test_query = "Sequence alignment algorithms using neural networks"
command_output = None

if openai_key and database_url:
    try:
        print(f"🔗 실제 DB RAG 파이프라인에 연결하여 툴 '{search_cs_papers.name}'을 실행합니다...")
        # 툴 함수 직접 비동기 호출 (Query, Mock Runtime 주입)
        command_output = await search_cs_papers.ainvoke(
            {"query": test_query, "runtime": mock_runtime, "k": 2}
        )
        print("✅ 툴 실행 결과 Command 생성 완료.")
    except Exception as e:
        print(f"⚠️ 실제 DB 툴 실행 실패 ({e}). 시뮬레이션용 Mock Command를 생성합니다.")
else:
    print("⚠️ API Key 또는 Database URL 설정이 누락되었습니다. 시뮬레이션용 Mock Command를 생성합니다.")

if command_output is None:
    # 시뮬레이션용 Mock Command 생성 및 바인딩
    mock_messages = [
        ToolMessage(
            content=(
                "검색 결과: 'Sequence alignment algorithms using neural networks' (cs.NE 논문)\n"
                "================================================================================\n"
                "\n"
                "[논문 1] (유사도: 0.8850)\n"
                "제목: Aligning Sequences with Neural Networks\n"
                "arXiv ID: 2104.12345\n"
                "\n"
                "내용:\n"
                "In this work, we present a novel approach to sequence alignment using deep neural networks...\n"
                "--------------------------------------------------------------------------------\n"
            ),
            tool_call_id=mock_runtime.tool_call_id
        )
    ]
    mock_sources = [
        {
            "arxiv_id": "2104.12345",
            "title": "Aligning Sequences with Neural Networks",
            "summary": "In this work, we present a novel approach to sequence alignment using deep neural networks..."
        }
    ]
    command_output = Command(update={
        "messages": mock_messages,
        "sources": mock_sources,
    })


🔗 실제 DB RAG 파이프라인에 연결하여 툴 'search_cs_papers'을 실행합니다...
✅ 툴 실행 결과 Command 생성 완료.


## 4단계. 반환된 Command 내부 구조 분석

LangGraph 툴이 에이전트 상태 업데이트를 위해 반환한 `Command` 내부의 구체적인 페이로드(Payload) 구조를 뜯어봅니다.

In [13]:
# Command 객체로부터 업데이트 딕셔너리 추출
update_data = command_output.update or {}

print("========================================================================")
print(" 💡 LangGraph Command(update={...}) 상태 갱신 데이터 상세 분석")
print("========================================================================")

# 1. messages 상태 업데이트 확인 (대화 이력에 툴 응답 누적)
tool_messages = update_data.get("messages", [])
if tool_messages:
    msg = tool_messages[0]
    print(f" 1) [messages] 업데이트 항목 감지 : {type(msg).__name__}")
    print(f"    - 대응되는 Tool Call ID : '{msg.tool_call_id}'")
    print(f"    - 툴 응답 본문 일부     : \"{msg.content[:95].replace('\n', ' ')}...\"")
else:
    print(" 1) [messages] 업데이트 데이터 누락")

print("------------------------------------------------------------------------")

# 2. sources 상태 업데이트 확인 (UI 및 RAG 인용 출처 표시용)
sources_data = update_data.get("sources", [])
print(f" 2) [sources] 인용 출처 항목 감지 : 총 {len(sources_data)}건")
for i, src in enumerate(sources_data, 1):
    print(f"    * 출처 #{i}")
    print(f"      - arXiv ID : {src.get('arxiv_id')}")
    print(f"      - 논문 제목 : {src.get('title')}")
    print(f"      - 요약 스니펫: \"{src.get('summary')}\"")

print("========================================================================")

 💡 LangGraph Command(update={...}) 상태 갱신 데이터 상세 분석
 1) [messages] 업데이트 항목 감지 : ToolMessage
    - 대응되는 Tool Call ID : 'call_mock_12345'
    - 툴 응답 본문 일부     : "검색 결과: 'Sequence alignment algorithms using neural networks' (cs.NE 논문)  ======================..."
------------------------------------------------------------------------
 2) [sources] 인용 출처 항목 감지 : 총 2건
    * 출처 #1
      - arXiv ID : 1708.09097
      - 논문 제목 : Optimizing scoring function of dynamic programming of pairwise profile   alignment using derivative free neural network
      - 요약 스니펫: "Title: Optimizing scoring function of dynamic programming of pairwise profile alignment using derivative free neural network Abstract: A profile comparison meth…"
    * 출처 #2
      - arXiv ID : 2004.02094
      - 논문 제목 : Locality Sensitive Hashing-based Sequence Alignment Using Deep   Bidirectional LSTM Models
      - 요약 스니펫: "Title: Locality Sensitive Hashing-based Sequence Alignment Using Deep Bidirectional LSTM Models Abstract: Bidir

## 결론 및 정리

본 심화 가이드를 통해 다음 핵심 연동 메커니즘을 마스터하였습니다:
* **`ToolRuntime` 매핑** : LangGraph 세션 내에서 LLM의 병렬 툴 호출 ID(`tool_call_id`)와 툴의 응답 메시지(`ToolMessage`)를 정확히 결합시켜 줍니다.
* **`Command` 객체의 역할** : 단순한 텍스트 리턴이 아닌, 그래프 상태 채널의 데이터를 갱신(Update)하기 위한 흐름 제어 도구입니다. 이를 통해 챗봇 메인 메시지 창 외에 **"참고한 출처 리스트"** 등의 부가 UI 위젯 정보를 에이전트가 동시에 관리할 수 있습니다.